In [ ]:
# ============================================================================# FAILSAFE: Force Path Correction and Package Installation# ============================================================================import sysimport subprocessimport osfrom pathlib import Pathfrom datetime import datetime
import shutil
def force_install_package(package_name, import_name=None):    """Force install package using multiple methods."""    if import_name is None:        import_name = package_name.split('[')[0].split('==')[0].split('>=')[0]        # Try import first    try:        __import__(import_name)        return True    except ImportError:        pass        # Method 1: pip install --user    try:        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', '--quiet', package_name],                              stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)        __import__(import_name)        return True    except:        pass        # Method 2: pip install --break-system-packages (Python 3.12+)    try:        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--break-system-packages', '--quiet', package_name],                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)        __import__(import_name)        return True    except:        pass        # Method 3: pip install system-wide    try:        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', package_name],                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)        __import__(import_name)        return True    except:        pass        # Method 4: conda install (if conda available)    try:        subprocess.check_call(['conda', 'install', '-y', '--quiet', package_name],                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)        __import__(import_name)        return True    except:        pass        # Method 5: apt-get install (Linux/Docker)    if os.path.exists('/usr/bin/apt-get'):        try:            apt_package = f'python3-{import_name.replace("_", "-")}'            subprocess.check_call(['apt-get', 'install', '-y', '--quiet', apt_package],                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)            __import__(import_name)            return True        except:            pass        # Method 6: Direct pip install with --force-reinstall    try:        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--force-reinstall', '--quiet', package_name],                             stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)        __import__(import_name)        return True    except:        pass        print(f"⚠️  Warning: Could not install {package_name}, continuing anyway...")    return Falsedef correct_file_path(file_path, search_paths=None):    """Correct file path by searching multiple locations."""    if isinstance(file_path, str):        file_path = Path(file_path)        # If path exists, return it    if file_path.exists():        return file_path        # Default search paths    if search_paths is None:        search_paths = [            Path.cwd(),            Path('/workspace/client/db'),            Path('/workspace/db'),            Path('/workspace'),            Path('/content/drive/MyDrive/db'),            Path('/content/db'),            Path('/content'),            Path.home() / 'Documents' / 'AQ' / 'db',            BASE_DIR if 'BASE_DIR' in globals() else Path('/Users/machine/Documents/AQ/db'),        ]        # Search recursively    for search_path in search_paths:        if not search_path.exists():            continue                # Try direct path        candidate = search_path / file_path.name        if candidate.exists():            return candidate                # Try recursive search        try:            for found_path in search_path.rglob(file_path.name):                if found_path.is_file():                    return found_path        except:            continue        # Return original path (will fail later, but at least we tried)    return file_pathdef create_notebook_backup(notebook_path=None):
    """Create backup of current notebook automatically."""
    try:
        # Try to detect notebook path from various sources
        if notebook_path is None:
            # Try to get from __file__ or current working directory
            try:
                notebook_path = Path(__file__)
            except:
                notebook_path = Path.cwd() / 'current_notebook.ipynb'
        
        if isinstance(notebook_path, str):
            notebook_path = Path(notebook_path)
        
        # Only create backup if file exists
        if notebook_path.exists() and notebook_path.suffix == '.ipynb':
            timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
            backup_path = notebook_path.parent / f"{notebook_path.stem}_{timestamp}.backup.ipynb"
            
            # Create backup
            shutil.copy2(notebook_path, backup_path)
            print(f"✅ Backup created: {backup_path.name}")
            return backup_path
        else:
            print("⚠️  Could not determine notebook path for backup")
            return None
    except Exception as e:
        print(f"⚠️  Backup creation failed (non-critical): {e}")
        return None

# Create backup at startup
try:
    create_notebook_backup()
except Exception as e:
    print(f"⚠️  Backup skipped: {e}")
def ensure_packages_installed():    """Ensure all required packages are installed."""    required_packages = [        ('psycopg2-binary', 'psycopg2'),        ('pandas', 'pandas'),        ('numpy', 'numpy'),        ('matplotlib', 'matplotlib'),        ('seaborn', 'seaborn'),        ('ipython', 'IPython'),        ('jupyter', 'jupyter'),    ]        print("\n" + "="*80)    print("FAILSAFE: Ensuring all packages are installed...")    print("="*80)        for package, import_name in required_packages:        if force_install_package(package, import_name):            print(f"✅ {package} installed")        else:            print(f"⚠️  {package} installation failed, but continuing...")        print("="*80 + "\n")def ensure_paths_correct():
    """Ensure all file paths are correct."""
    print("\n" + "="*80)
    print("FAILSAFE: Correcting file paths...")
    print("="*80)
    
    # Correct BASE_DIR if needed - fix UnboundLocalError
    base_dir_exists = 'BASE_DIR' in globals()
    base_dir_valid = False
    
    if base_dir_exists:
        try:
            base_dir_value = globals()['BASE_DIR']
            if base_dir_value:
                base_dir_path = Path(base_dir_value) if isinstance(base_dir_value, str) else base_dir_value
                base_dir_valid = base_dir_path.exists()
        except:
            base_dir_valid = False
    
    if not base_dir_exists or not base_dir_valid:
        corrected_base_dir = correct_file_path(Path('/Users/machine/Documents/AQ/db'))
        globals()['BASE_DIR'] = corrected_base_dir
        print(f"✅ BASE_DIR corrected: {corrected_base_dir}")
    else:
        print(f"✅ BASE_DIR valid: {globals()['BASE_DIR']}")
    
    # Correct DB_DIR if needed - fix UnboundLocalError
    db_dir_exists = 'DB_DIR' in globals()
    db_dir_valid = False
    db_dir_value = None
    
    if db_dir_exists:
        try:
            db_dir_value = globals()['DB_DIR']
            if db_dir_value:
                db_dir_path = Path(db_dir_value) if isinstance(db_dir_value, str) else db_dir_value
                db_dir_valid = db_dir_path.exists()
        except:
            db_dir_valid = False
    
    if db_dir_exists and db_dir_value and not db_dir_valid:
        db_dir_path = Path(db_dir_value) if isinstance(db_dir_value, str) else db_dir_value
        corrected_db_dir = correct_file_path(db_dir_path)
        globals()['DB_DIR'] = corrected_db_dir
        print(f"✅ DB_DIR corrected: {corrected_db_dir}")
    elif db_dir_exists and db_dir_value:
        print(f"✅ DB_DIR valid: {globals()['DB_DIR']}")
    
    print("="*80 + "\n")
# Run failsafe checksensure_packages_installed()ensure_paths_correct()print("✅ Failsafe checks complete")

# DB-12: Credit Card Optimization Database - Query Testing & Documentation

This notebook provides comprehensive testing, documentation, and visualization for all SQL queries.

## Database Overview

**Database Name:** Credit Card Optimization Database  
**Database ID:** db-12  
**Domain:** Credit Card Optimization  
**Total Queries:** 30  

## Workflow

1. Database initialization (create database, load schema, load data)
2. Query execution and validation
3. Results visualization and analysis
4. Performance metrics and documentation

In [ ]:
# ============================================================================# STREAMLIT DASHBOARD EXECUTION# ============================================================================import subprocessimport sysimport osfrom pathlib import Pathimport webbrowserimport timeimport threadingdef find_dashboard_file():    """Find Streamlit dashboard file recursively."""    search_paths = [        Path.cwd(),        Path('/workspace/client/db'),        Path('/workspace/db'),        Path('/workspace'),        Path('/content/drive/MyDrive/db'),        Path('/content/db'),        Path('/content'),        Path.home() / 'Documents' / 'AQ' / 'db',    ]        dashboard_name = f'{DB_NAME}_dashboard.py'        for search_path in search_paths:        if not search_path.exists():            continue                # Try direct path        candidate = search_path / dashboard_name        if candidate.exists():            return candidate                # Try recursive search        try:            for found_path in search_path.rglob(dashboard_name):                if found_path.is_file():                    return found_path        except:            continue        return Nonedef run_streamlit_dashboard(method='notebook', port=8501, open_browser=True):    """    Run Streamlit dashboard from Jupyter notebook.        Methods:    - 'notebook': Run in notebook output (using streamlit's notebook mode)    - 'subprocess': Run as subprocess (background)    - 'magic': Use !streamlit run magic command    """    dashboard_path = find_dashboard_file()        if not dashboard_path:        print("❌ Dashboard file not found")        print(f"   Looking for: {DB_NAME}_dashboard.py")        return None        print(f"✅ Found dashboard: {dashboard_path}")        if method == 'notebook':        # Method 1: Run Streamlit in notebook-compatible mode        # Note: Streamlit doesn't natively support notebooks, but we can use iframe        print("\n" + "="*80)        print("STREAMLIT DASHBOARD - NOTEBOOK MODE")        print("="*80)        print(f"\nDashboard: {dashboard_path.name}")        print(f"\nTo run dashboard:")        print(f"  1. Run this cell to start the server")        print(f"  2. Open the URL shown below in a new tab")        print(f"  3. Or use: !streamlit run {dashboard_path} --server.port={port}")        print("\n" + "="*80)                # Start Streamlit as subprocess        cmd = [            sys.executable, '-m', 'streamlit', 'run',            str(dashboard_path),            '--server.port', str(port),            '--server.headless', 'true',            '--server.runOnSave', 'false',            '--browser.gatherUsageStats', 'false'        ]                process = subprocess.Popen(            cmd,            stdout=subprocess.PIPE,            stderr=subprocess.PIPE,            text=True        )                # Wait a moment for server to start        time.sleep(2)                # Get the URL        url = f"http://localhost:{port}"        print(f"\n🌐 Dashboard URL: {url}")        print(f"\nServer started in background (PID: {process.pid})")        print(f"\nTo stop: process.terminate() or run stop_streamlit()")                # Store process for later termination        globals()['_streamlit_process'] = process                # Try to open browser        if open_browser:            try:                webbrowser.open(url)            except:                pass                return process        elif method == 'subprocess':        # Method 2: Run as background subprocess        cmd = [            sys.executable, '-m', 'streamlit', 'run',            str(dashboard_path),            '--server.port', str(port)        ]                process = subprocess.Popen(cmd)        print(f"✅ Streamlit started (PID: {process.pid})")        print(f"🌐 Dashboard: http://localhost:{port}")        return process        elif method == 'magic':        # Method 3: Print magic command for user to run        print("Run this command in a new cell:")        print(f"!streamlit run {dashboard_path} --server.port={port}")        return Nonedef stop_streamlit():    """Stop running Streamlit process."""    if '_streamlit_process' in globals():        process = globals()['_streamlit_process']        process.terminate()        print("✅ Streamlit stopped")    else:        print("⚠️  No Streamlit process found")# Auto-detect DB_NAME if not setif 'DB_NAME' not in globals():    # Try to detect from current directory or notebook name    cwd = Path.cwd()    for db_num in range(6, 16):        if f'db-{db_num}' in str(cwd) or f'db{db_num}' in str(cwd):            DB_NAME = f'db-{db_num}'            break    else:        DB_NAME = 'db-6'  # Default        print(f"⚠️  Could not detect DB_NAME, using default: {DB_NAME}")print("\n" + "="*80)print("STREAMLIT DASHBOARD INTEGRATION")print("="*80)print(f"Database: {DB_NAME}")print("\nAvailable methods:")print("  1. run_streamlit_dashboard(method='notebook') - Run in notebook mode")print("  2. run_streamlit_dashboard(method='subprocess') - Run as background process")print("  3. run_streamlit_dashboard(method='magic') - Get magic command")print("  4. stop_streamlit() - Stop running dashboard")print("\n" + "="*80)

## Streamlit Dashboard

Run the Streamlit dashboard using one of these methods:

**Method 1: Notebook Mode** (Recommended)
```python
run_streamlit_dashboard(method='notebook', port=8501)
```

**Method 2: Magic Command**
```bash
!streamlit run db-12_dashboard.py --server.port=8501
```

**Method 3: Background Process**
```python
run_streamlit_dashboard(method='subprocess', port=8501)
```


In [ ]:
# Import required libraries
import psycopg2
import json
import os
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, Markdown
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Configuration
DB_NAME = 'db12'
DB_DIR = Path('/Users/machine/Documents/AQ/db/db-12')
BASE_DIR = Path('/Users/machine/Documents/AQ/db')

DB_CONFIG = {
    'host': os.getenv('PG_HOST', 'localhost'),
    'port': os.getenv('PG_PORT', '5432'),
    'user': os.getenv('PG_USER', os.getenv('USER', 'machine')),
    'password': os.getenv('PG_PASSWORD', ''),
    'database': 'postgres'
}

print(f"Database: {DB_NAME}")
print(f"Database Directory: {DB_DIR}")
print(f"Configuration: {DB_CONFIG['host']}:{DB_CONFIG['port']}")

## Step 1: Database Initialization

In [ ]:
def initialize_database(db_name: str, db_dir: Path, config: dict) -> bool:
    """Initialize database: create, load schema, load data."""
    try:
        # Connect to postgres database
        conn = psycopg2.connect(**config)
        conn.autocommit = True
        cur = conn.cursor()
        
        # Check if database exists
        cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (db_name,))
        exists = cur.fetchone()
        
        if not exists:
            cur.execute(f'CREATE DATABASE {db_name}')
            print(f"✅ Created database: {db_name}")
        else:
            print(f"ℹ️  Database {db_name} already exists")
        
        cur.close()
        conn.close()
        
        # Load schema
        schema_file = db_dir / 'data' / 'schema.sql'
        if schema_file.exists():
            db_config = config.copy()
            db_config['database'] = db_name
            conn = psycopg2.connect(**db_config)
            cur = conn.cursor()
            
            with open(schema_file, 'r') as f:
                schema_sql = f.read()
            
            statements = [s.strip() for s in schema_sql.split(';') if s.strip()]
            for statement in statements:
                if statement:
                    try:
                        cur.execute(statement)
                    except Exception as e:
                        if 'already exists' not in str(e).lower():
                            pass
            
            conn.commit()
            cur.close()
            conn.close()
            print(f"✅ Loaded schema for {db_name}")
        
        # Load data
        data_file = db_dir / 'data' / 'data.sql'
        if data_file.exists():
            db_config = config.copy()
            db_config['database'] = db_name
            conn = psycopg2.connect(**db_config)
            cur = conn.cursor()
            
            with open(data_file, 'r') as f:
                data_sql = f.read()
            
            statements = [s.strip() for s in data_sql.split(';') if s.strip()]
            for statement in statements:
                if statement:
                    try:
                        cur.execute(statement)
                    except Exception as e:
                        if 'duplicate' not in str(e).lower():
                            pass
            
            conn.commit()
            cur.close()
            conn.close()
            print(f"✅ Loaded data for {db_name}")
        
        return True
    except Exception as e:
        print(f"❌ Error initializing database: {e}")
        return False

# Initialize database
print("="*60)
print("DATABASE INITIALIZATION")
print("="*60)
initialize_database(DB_NAME, DB_DIR, DB_CONFIG)

## Step 2: Load Query Metadata

In [ ]:
# Load queries from JSON
queries_file = DB_DIR / 'queries' / 'queries.json'

with open(queries_file) as f:
    queries_data = json.load(f)

queries = queries_data.get('queries', [])
total_queries = len(queries)

print(f"Loaded {total_queries} queries from {queries_file}")
print(f"\nQuery Overview:")
for q in queries[:5]:  # Show first 5
    print(f"  Query {q.get('number')}: {q.get('title', 'N/A')[:60]}...")
if total_queries > 5:
    print(f"  ... and {total_queries - 5} more queries")

## Step 3: Query Execution Function

In [ ]:
def execute_query_with_metrics(db_name: str, query_sql: str, query_num: int, config: dict):
    """Execute a query and return results with metrics."""
    db_config = config.copy()
    db_config['database'] = db_name
    
    start_time = datetime.now()
    
    try:
        conn = psycopg2.connect(**db_config)
        cur = conn.cursor()
        
        query_clean = query_sql.strip().rstrip(';')
        cur.execute(query_clean)
        
        columns = [desc[0] for desc in cur.description] if cur.description else []
        rows = cur.fetchall()
        
        df = pd.DataFrame(rows, columns=columns) if columns else pd.DataFrame()
        
        execution_time = (datetime.now() - start_time).total_seconds()
        
        cur.close()
        conn.close()
        
        return {
            'success': True,
            'error': None,
            'dataframe': df,
            'execution_time': execution_time,
            'row_count': len(df),
            'column_count': len(df.columns),
            'columns': columns
        }
        
    except Exception as e:
        execution_time = (datetime.now() - start_time).total_seconds()
        error_msg = str(e)
        
        if 'cur' in locals():
            cur.close()
        if 'conn' in locals():
            conn.close()
        
        return {
            'success': False,
            'error': error_msg,
            'dataframe': None,
            'execution_time': execution_time,
            'row_count': 0,
            'column_count': 0,
            'columns': []
        }

## Step 4: Execute All Queries

In [ ]:
# Execute all queries and collect results
all_results = []

print("="*60)
print("EXECUTING ALL QUERIES")
print("="*60)

for query_info in queries:
    query_num = query_info.get('number')
    query_sql = query_info.get('sql', '')
    query_title = query_info.get('title', f'Query {query_num}')
    
    result = execute_query_with_metrics(DB_NAME, query_sql, query_num, DB_CONFIG)
    result['query_number'] = query_num
    result['query_title'] = query_title
    result['query_info'] = query_info
    
    all_results.append(result)
    
    status = "✅" if result['success'] else "❌"
    print(f"{status} Query {query_num}: {query_title[:50]}... ({result['execution_time']:.3f}s, {result['row_count']} rows)")

# Summary
passed = sum(1 for r in all_results if r['success'])
failed = sum(1 for r in all_results if not r['success'])
print(f"\n{'='*60}")
print(f"SUMMARY: {passed}/{total_queries} passed ({passed/total_queries*100:.1f}%)")
print(f"{'='*60}")

## Step 5: Performance Visualization

In [ ]:
# Create performance metrics DataFrame
perf_data = []
for r in all_results:
    perf_data.append({
        'Query': r['query_number'],
        'Title': r['query_title'][:40] + '...' if len(r['query_title']) > 40 else r['query_title'],
        'Execution Time (s)': r['execution_time'],
        'Row Count': r['row_count'],
        'Column Count': r['column_count'],
        'Status': 'Passed' if r['success'] else 'Failed'
    })

perf_df = pd.DataFrame(perf_data)

# Visualization 1: Execution Time Distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Execution time bar chart
axes[0, 0].bar(perf_df['Query'], perf_df['Execution Time (s)'], color='steelblue', alpha=0.7)
axes[0, 0].set_xlabel('Query Number')
axes[0, 0].set_ylabel('Execution Time (seconds)')
axes[0, 0].set_title('Query Execution Time by Query Number')
axes[0, 0].tick_params(axis='x', rotation=45)
axes[0, 0].grid(True, alpha=0.3)

# Execution time histogram
axes[0, 1].hist(perf_df['Execution Time (s)'], bins=20, color='coral', alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('Execution Time (seconds)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Execution Times')
axes[0, 1].grid(True, alpha=0.3)

# Row count bar chart
axes[1, 0].bar(perf_df['Query'], perf_df['Row Count'], color='green', alpha=0.7)
axes[1, 0].set_xlabel('Query Number')
axes[1, 0].set_ylabel('Row Count')
axes[1, 0].set_title('Rows Returned by Query')
axes[1, 0].tick_params(axis='x', rotation=45)
axes[1, 0].grid(True, alpha=0.3)

# Status pie chart
status_counts = perf_df['Status'].value_counts()
axes[1, 1].pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%', startangle=90)
axes[1, 1].set_title('Query Execution Status')

plt.tight_layout()
plt.show()

# Display performance summary table
print("\nPerformance Summary:")
print(f"  Average execution time: {perf_df['Execution Time (s)'].mean():.3f}s")
print(f"  Median execution time: {perf_df['Execution Time (s)'].median():.3f}s")
print(f"  Max execution time: {perf_df['Execution Time (s)'].max():.3f}s")
print(f"  Min execution time: {perf_df['Execution Time (s)'].min():.3f}s")
print(f"  Total rows returned: {perf_df['Row Count'].sum():,}")
print(f"  Average rows per query: {perf_df['Row Count'].mean():.1f}")

## Step 6: Individual Query Documentation and Visualization

In [ ]:
def document_and_visualize_query(query_result: dict, query_num: int):
    """Create comprehensive documentation and visualization for a single query."""
    query_info = query_result['query_info']
    
    # Create markdown documentation
    doc = f"""
## Query {query_num}: {query_info.get('title', 'N/A')}

### Execution Status
- **Status:** {'✅ PASSED' if query_result['success'] else '❌ FAILED'}
- **Execution Time:** {query_result['execution_time']:.3f} seconds
- **Rows Returned:** {query_result['row_count']:,}
- **Columns Returned:** {query_result['column_count']}

### Query Information
- **Description:** {query_info.get('description', 'N/A')[:300]}...
- **Use Case:** {query_info.get('use_case', 'N/A')}
- **Business Value:** {query_info.get('business_value', 'N/A')}
- **Complexity:** {query_info.get('complexity', 'N/A')}
- **Expected Output:** {query_info.get('expected_output', 'N/A')}

### SQL Query
```sql
{query_info.get('sql', '')[:1000]}...
```

### Results Preview
"""
    
    display(Markdown(doc))
    
    if query_result['success'] and query_result['dataframe'] is not None:
        df = query_result['dataframe']
        
        if len(df) > 0:
            print(f"\nFirst 10 rows of Query {query_num}:")
            display(df.head(10))
            
            # Create visualizations if numeric data exists
            numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
            if len(numeric_cols) > 0:
                num_plots = min(3, len(numeric_cols))
                fig, axes = plt.subplots(1, num_plots, figsize=(15, 4))
                if num_plots == 1:
                    axes = [axes]
                
                for idx, col in enumerate(numeric_cols[:num_plots]):
                    if df[col].notna().sum() > 0:
                        axes[idx].hist(df[col].dropna(), bins=min(20, len(df)), alpha=0.7, edgecolor='black')
                        axes[idx].set_title(f'Distribution of {col[:30]}')
                        axes[idx].set_xlabel(col[:30])
                        axes[idx].set_ylabel('Frequency')
                        axes[idx].grid(True, alpha=0.3)
                
                plt.tight_layout()
                plt.show()
                
                # Create correlation heatmap if multiple numeric columns
                if len(numeric_cols) > 1:
                    fig, ax = plt.subplots(figsize=(10, 8))
                    corr_matrix = df[numeric_cols].corr()
                    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
                    ax.set_title('Correlation Matrix of Numeric Columns')
                    plt.tight_layout()
                    plt.show()
        else:
            print(f"\nQuery {query_num} returned 0 rows.")
    else:
        if query_result.get('error'):
            print(f"\n❌ Error: {query_result['error'][:200]}")

# Document and visualize each query
for query_result in all_results:
    query_num = query_result['query_number']
    document_and_visualize_query(query_result, query_num)
    print("\n" + "="*80 + "\n")

## Step 7: Generate Comprehensive Report

In [ ]:
# Create comprehensive report
report_data = {
    'database': DB_NAME,
    'test_timestamp': datetime.now().isoformat(),
    'total_queries': total_queries,
    'passed': passed,
    'failed': failed,
    'success_rate': passed / total_queries * 100,
    'average_execution_time': perf_df['Execution Time (s)'].mean(),
    'total_execution_time': perf_df['Execution Time (s)'].sum(),
    'queries': []
}

for r in all_results:
    query_report = {
        'number': r['query_number'],
        'title': r['query_title'],
        'success': r['success'],
        'execution_time': r['execution_time'],
        'row_count': r['row_count'],
        'column_count': r['column_count'],
        'columns': r['columns']
    }
    if not r['success']:
        query_report['error'] = r['error']
    
    report_data['queries'].append(query_report)

# Save report
report_file = DB_DIR / 'results' / f'{DB_NAME}_comprehensive_report.json'
report_file.parent.mkdir(exist_ok=True)

with open(report_file, 'w') as f:
    json.dump(report_data, f, indent=2, default=str)

print(f"✅ Comprehensive report saved to: {report_file}")

# Display summary
print("\n" + "="*60)
print("COMPREHENSIVE TEST REPORT")
print("="*60)
print(f"Database: {DB_NAME}")
print(f"Total Queries: {total_queries}")
print(f"Passed: {passed}")
print(f"Failed: {failed}")
print(f"Success Rate: {passed/total_queries*100:.1f}%")
print(f"Average Execution Time: {perf_df['Execution Time (s)'].mean():.3f}s")
print(f"Total Execution Time: {perf_df['Execution Time (s)'].sum():.3f}s")